# Exchangeability Analysis Notebook
Load and inspect metrics produced by `scripts/analyze_exchangeability.py`.

In [ ]:
import os
from pathlib import Path
import pandas as pd

In [ ]:
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', 'outputs'))
candidates = [
    BASE_SAVE_DIR / 'exchangeability_metrics_w32.csv',
    BASE_SAVE_DIR / 'exchangeability_metrics.csv',
    Path('outputs/exchangeability_metrics_w32.csv'),
    Path('outputs/exchangeability_metrics.csv'),
]
CSV_PATH = next((p for p in candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('No exchangeability metrics CSV found in expected locations.')
print(f'Using CSV_PATH={CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(f'rows={len(df)}, widths={sorted(df["width"].unique().tolist())}')
df.head()

In [ ]:
coverage = (
    df.groupby(['width', 'representation', 'analysis_type'], as_index=False)
      .agg(num_points=('images_seen', 'nunique'), min_images_seen=('images_seen', 'min'), max_images_seen=('images_seen', 'max'))
      .sort_values(['width', 'representation', 'analysis_type'])
)
coverage

In [ ]:
summary = (
    df.groupby(['representation', 'analysis_type', 'width', 'images_seen'], as_index=False)
      .agg(
          ks_distance_mean=('ks_distance', 'mean'),
          w1_distance_mean=('w1_distance', 'mean'),
          train_loss=('train_loss', 'mean'),
          val_loss=('val_loss', 'mean'),
          train_error=('train_error', 'mean'),
          val_error=('val_error', 'mean'),
      )
      .sort_values(['width', 'images_seen', 'representation', 'analysis_type'])
)
summary.head()

In [ ]:
summary_path = CSV_PATH.with_name(CSV_PATH.stem + '_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Wrote {summary_path}')